### Advanced Context Managers Project Solution

In this notebook we will build a small but realistic project using Python context managers.

The goal is not just to use `with open(...)`.

The goal is to design context managers that make a workflow safer, cleaner, and easier to reason about.

We will build a **transactional report pipeline**.

The pipeline will:

1. Create a temporary working directory
2. Temporarily change environment variables
3. Temporarily change the current working directory
4. Write output files atomically
5. Open multiple files safely
6. Measure how long each phase takes
7. Roll back work if an error occurs
8. Clean up resources automatically

This is the same general pattern context managers are designed for:

* Setup - Cleanup
* Open - Close
* Change - Reset
* Begin - Commit / Rollback
* Enter - Exit

First, let's import everything we will need.

We will only use Python's standard library.

In [1]:
from __future__ import annotations

import contextlib
import csv
import os
import tempfile
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable, Iterator, Optional, TextIO


Let's create some small input data.

In a real project this might come from a database, an API, or a file.

For this project, a list of dictionaries is enough.

In [2]:
SALES = [
    {"region": "North", "product": "Keyboard", "units": 12, "unit_price": 49.99},
    {"region": "North", "product": "Mouse", "units": 18, "unit_price": 24.99},
    {"region": "South", "product": "Monitor", "units": 5, "unit_price": 199.99},
    {"region": "South", "product": "Keyboard", "units": 7, "unit_price": 49.99},
    {"region": "East", "product": "Mouse", "units": 21, "unit_price": 24.99},
    {"region": "West", "product": "Monitor", "units": 3, "unit_price": 199.99},
]

SALES[:2]

[{'region': 'North', 'product': 'Keyboard', 'units': 12, 'unit_price': 49.99},
 {'region': 'North', 'product': 'Mouse', 'units': 18, 'unit_price': 24.99}]

Before writing context managers, let's write one helper function.

This function summarizes sales by region.

In [3]:
def summarize_sales(rows: Iterable[dict]) -> dict[str, float]:
    totals: dict[str, float] = {}

    for row in rows:
        region = row["region"]
        revenue = row["units"] * row["unit_price"]
        totals[region] = totals.get(region, 0.0) + revenue

    return totals

summary = summarize_sales(SALES)
summary

{'North': 1049.7, 'South': 1349.88, 'East': 524.79, 'West': 599.97}

Now we will start building the project.

The first context manager will measure elapsed time.

It will follow this pattern:

1. Store the start time in `__enter__`
2. Store the end time in `__exit__`
3. Do not suppress exceptions

In [4]:
@dataclass
class Timer:
    label: str
    start: float = field(init=False, default=0.0)
    end: float = field(init=False, default=0.0)
    elapsed: float = field(init=False, default=0.0)

    def __enter__(self) -> "Timer":
        print(f"starting: {self.label}")
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        self.end = time.perf_counter()
        self.elapsed = self.end - self.start
        print(f"finished: {self.label} in {self.elapsed:.6f} seconds")
        return False

Let's test it on a tiny piece of code.

In [5]:
with Timer("sales summary") as timer:
    result = summarize_sales(SALES)

print(result)
print(timer.elapsed > 0)

starting: sales summary
finished: sales summary in 0.000015 seconds
{'North': 1049.7, 'South': 1349.88, 'East': 524.79, 'West': 599.97}
True


Notice that `timer` still exists after the `with` block.

The `with` statement does not create a local scope.

This is useful here because the object returned by `__enter__` stores the elapsed time after the block finishes.

Next, let's create a context manager that temporarily changes an environment variable.

This is an example of the pattern:

**Change - Reset**

When we enter the context, we change something.

When we exit the context, we restore the original value.

In [6]:
@dataclass
class temporary_env:
    key: str
    value: str
    old_value: Optional[str] = field(init=False, default=None)
    existed_before: bool = field(init=False, default=False)

    def __enter__(self) -> str:
        self.existed_before = self.key in os.environ
        self.old_value = os.environ.get(self.key)
        os.environ[self.key] = self.value
        return self.value

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if self.existed_before:
            os.environ[self.key] = self.old_value or ""
        else:
            os.environ.pop(self.key, None)

        return False

Let's make sure it resets the environment variable correctly.

In [7]:
print("before:", os.environ.get("REPORT_MODE"))

with temporary_env("REPORT_MODE", "debug") as mode:
    print("inside:", os.environ.get("REPORT_MODE"))
    print("returned:", mode)

print("after:", os.environ.get("REPORT_MODE"))

before: None
inside: debug
returned: debug
after: None


Now let's create a context manager that temporarily changes the current working directory.

This is another **Change - Reset** pattern.

It is very important to reset the working directory even if something fails inside the block.

In [8]:
@dataclass
class change_dir:
    path: Path
    old_path: Path = field(init=False)

    def __enter__(self) -> Path:
        self.old_path = Path.cwd()
        os.chdir(self.path)
        return self.path

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        os.chdir(self.old_path)
        return False

Let's test it with a temporary directory.

The directory is created by Python's built-in `TemporaryDirectory` context manager.

In [9]:
original = Path.cwd()

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)

    with change_dir(tmp_path):
        print("inside:", Path.cwd() == tmp_path)

print("after:", Path.cwd() == original)

inside: True
after: True


Now let's solve one of the most common production problems:

**How do we write a file safely?**

If a program crashes halfway through writing a file, we might leave behind a corrupted partial file.

A better approach is:

1. Write to a temporary file
2. If everything succeeds, replace the real file with the temporary file
3. If anything fails, delete the temporary file

This is called an atomic write pattern.

In [10]:
@dataclass
class atomic_write:
    target: Path
    mode: str = "w"
    encoding: str = "utf-8"
    temp_path: Path = field(init=False)
    file: TextIO | None = field(init=False, default=None)

    def __enter__(self) -> TextIO:
        self.target = Path(self.target)
        self.target.parent.mkdir(parents=True, exist_ok=True)

        suffix = self.target.suffix + ".tmp"
        self.temp_path = self.target.with_suffix(suffix)

        self.file = open(self.temp_path, self.mode, encoding=self.encoding, newline="")
        return self.file

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if self.file is not None:
            self.file.close()

        if exc_type is None:
            self.temp_path.replace(self.target)
        else:
            self.temp_path.unlink(missing_ok=True)

        return False


Let's test the successful case.

If no exception happens inside the block, the temporary file should become the final file.

In [11]:
with tempfile.TemporaryDirectory() as tmp:
    report_path = Path(tmp) / "report.txt"

    with atomic_write(report_path) as f:
        f.write("hello from an atomic file\n")

    print(report_path.exists())
    print(report_path.read_text())


True
hello from an atomic file



Now let's test the failure case.

If an exception happens inside the block, the final file should not be replaced by bad partial data.

In [12]:
with tempfile.TemporaryDirectory() as tmp:
    report_path = Path(tmp) / "report.txt"
    report_path.write_text("original content\n", encoding="utf-8")

    try:
        with atomic_write(report_path) as f:
            f.write("broken new content\n")
            raise RuntimeError("something failed while writing")
    except RuntimeError as ex:
        print("caught:", ex)

    print(report_path.read_text())


caught: something failed while writing
original content



The original file survived.

That is the kind of cleanup behavior context managers are very good at expressing.

Next, let's write a function that creates a CSV report.

This function does not need to know about temporary files or atomic writes.

It only needs a file-like object.

In [13]:
def write_csv_summary(file_obj, totals: dict[str, float]) -> None:
    writer = csv.writer(file_obj)
    writer.writerow(["region", "revenue"])

    for region, revenue in sorted(totals.items()):
        writer.writerow([region, f"{revenue:.2f}"])

Let's use `atomic_write` to safely create the CSV file.

In [14]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "summary.csv"

    with atomic_write(path) as f:
        write_csv_summary(f, summarize_sales(SALES))

    print(path.read_text())

region,revenue
East,524.79
North,1049.70
South,1349.88
West,599.97



Now let's add a log file.

We want to open both the report file and the log file safely.

For a fixed number of files, nested `with` statements are fine.

In [15]:
with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    report_path = tmp_path / "summary.csv"
    log_path = tmp_path / "pipeline.log"

    with atomic_write(report_path) as report_file:
        with atomic_write(log_path) as log_file:
            log_file.write("starting pipeline\n")
            write_csv_summary(report_file, summarize_sales(SALES))
            log_file.write("finished pipeline\n")

    print(report_path.read_text())
    print(log_path.read_text())


region,revenue
East,524.79
North,1049.70
South,1349.88
West,599.97

starting pipeline
finished pipeline



Nested `with` statements are clear, but if we need to manage a dynamic number of context managers, nesting becomes awkward.

Python gives us `contextlib.ExitStack` for this.

`ExitStack` lets us enter context managers programmatically and guarantees they are exited in reverse order.

In [16]:
with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    paths = [tmp_path / "a.txt", tmp_path / "b.txt", tmp_path / "c.txt"]

    with contextlib.ExitStack() as stack:
        files = [stack.enter_context(atomic_write(path)) for path in paths]

        for index, f in enumerate(files, start=1):
            f.write(f"file number {index}\n")

    for path in paths:
        print(path.name, "=>", path.read_text().strip())


a.txt => file number 1
b.txt => file number 2
c.txt => file number 3


Now let's create a higher-level project context manager.

This manager will represent one pipeline run.

When it enters:

1. It creates a workspace
2. It temporarily changes into that workspace
3. It sets an environment variable
4. It starts a timer

When it exits:

1. It exits all those context managers in the correct order
2. It cleans everything up
3. It does not suppress exceptions

This is a perfect use case for `ExitStack`.

In [17]:
@dataclass
class PipelineRun:
    name: str
    mode: str = "production"
    stack: contextlib.ExitStack = field(init=False)
    workspace_manager: tempfile.TemporaryDirectory = field(init=False)
    workspace: Path = field(init=False)
    timer: Timer = field(init=False)

    def __enter__(self) -> "PipelineRun":
        self.stack = contextlib.ExitStack()
        self.stack.__enter__()

        self.workspace_manager = self.stack.enter_context(tempfile.TemporaryDirectory())
        self.workspace = Path(self.workspace_manager)

        self.stack.enter_context(change_dir(self.workspace))
        self.stack.enter_context(temporary_env("REPORT_MODE", self.mode))
        self.timer = self.stack.enter_context(Timer(self.name))

        print("workspace:", self.workspace)
        print("mode:", os.environ.get("REPORT_MODE"))

        return self

    def path(self, filename: str) -> Path:
        return self.workspace / filename

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if exc_type:
            print(f"pipeline failed: {exc_type.__name__}: {exc_value}")
        else:
            print("pipeline succeeded")

        self.stack.__exit__(exc_type, exc_value, traceback)
        return False

Let's run the pipeline successfully.

In [18]:
with PipelineRun("daily sales report", mode="debug") as run:
    with atomic_write(run.path("summary.csv")) as report_file:
        write_csv_summary(report_file, summarize_sales(SALES))

    with atomic_write(run.path("pipeline.log")) as log_file:
        log_file.write("created summary.csv\n")
        log_file.write(f"mode was {os.environ.get('REPORT_MODE')}\n")

    print("files inside workspace:", sorted(p.name for p in run.workspace.iterdir()))
    print(run.path("summary.csv").read_text())

print("REPORT_MODE after pipeline:", os.environ.get("REPORT_MODE"))


starting: daily sales report
workspace: C:\Users\user1\AppData\Local\Temp\tmpmsgdc7bg
mode: debug
files inside workspace: ['pipeline.log', 'summary.csv']
region,revenue
East,524.79
North,1049.70
South,1349.88
West,599.97

pipeline succeeded
finished: daily sales report in 0.009377 seconds
REPORT_MODE after pipeline: None


Notice that the workspace was temporary.

After the context exits, it is cleaned up automatically.

That is good for temporary processing, but sometimes we want to keep the final output.

Let's create a second version of the pipeline that copies final files to a destination directory on success.

In [19]:
@dataclass
class PublishingPipelineRun:
    name: str
    destination: Path
    mode: str = "production"
    stack: contextlib.ExitStack = field(init=False)
    workspace: Path = field(init=False)
    timer: Timer = field(init=False)

    def __enter__(self) -> "PublishingPipelineRun":
        self.stack = contextlib.ExitStack()
        self.stack.__enter__()

        workspace_name = self.stack.enter_context(tempfile.TemporaryDirectory())
        self.workspace = Path(workspace_name)

        self.stack.enter_context(change_dir(self.workspace))
        self.stack.enter_context(temporary_env("REPORT_MODE", self.mode))
        self.timer = self.stack.enter_context(Timer(self.name))

        self.destination.mkdir(parents=True, exist_ok=True)
        return self

    def path(self, filename: str) -> Path:
        return self.workspace / filename

    def publish(self, filename: str) -> None:
        source = self.path(filename)
        target = self.destination / filename
        target.write_bytes(source.read_bytes())

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if exc_type:
            print("not publishing because the pipeline failed")
        else:
            print("publishing completed files")
            for file in self.workspace.iterdir():
                if file.is_file():
                    self.publish(file.name)

        self.stack.__exit__(exc_type, exc_value, traceback)
        return False

Let's test the publishing pipeline.

On success, files from the temporary workspace are copied into the destination folder.

In [20]:
with tempfile.TemporaryDirectory() as output_dir:
    destination = Path(output_dir) / "published"

    with PublishingPipelineRun("publish sales report", destination, mode="production") as run:
        with atomic_write(run.path("summary.csv")) as report_file:
            write_csv_summary(report_file, summarize_sales(SALES))

        with atomic_write(run.path("README.txt")) as readme:
            readme.write("This folder contains the generated sales report.\n")

    print(sorted(p.name for p in destination.iterdir()))
    print((destination / "summary.csv").read_text())


starting: publish sales report
publishing completed files
finished: publish sales report in 0.029013 seconds
['README.txt', 'summary.csv']
region,revenue
East,524.79
North,1049.70
South,1349.88
West,599.97



Now let's test the failure case.

If the pipeline fails, it should not publish partial output.

In [21]:
with tempfile.TemporaryDirectory() as output_dir:
    destination = Path(output_dir) / "published"

    try:
        with PublishingPipelineRun("broken report", destination) as run:
            with atomic_write(run.path("summary.csv")) as report_file:
                write_csv_summary(report_file, summarize_sales(SALES))

            raise ValueError("fake failure before publishing")

    except ValueError as ex:
        print("caught:", ex)

    print("destination exists:", destination.exists())
    print("published files:", list(destination.iterdir()) if destination.exists() else [])

starting: broken report
not publishing because the pipeline failed
finished: broken report in 0.003710 seconds
caught: fake failure before publishing
destination exists: True
published files: []


The destination directory exists because the pipeline created it.

But no files were published because the pipeline failed before the successful exit path.

This is exactly the kind of behavior we want from a transaction-like workflow.

Let's improve the solution by adding a small validation context manager.

This context manager does not create a resource.

Instead, it checks a condition when leaving the block.

That can be useful when you want to verify that a workflow produced the expected outputs.

In [22]:
@dataclass
class require_files:
    folder: Path
    filenames: list[str]

    def __enter__(self) -> "require_files":
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if exc_type is not None:
            return False

        missing = [name for name in self.filenames if not (self.folder / name).exists()]

        if missing:
            raise FileNotFoundError(f"missing required files: {missing}")

        print("all required files exist")
        return False

Let's use `require_files` with the publishing pipeline.

In [23]:
with tempfile.TemporaryDirectory() as output_dir:
    destination = Path(output_dir) / "published"

    with PublishingPipelineRun("validated report", destination) as run:
        with atomic_write(run.path("summary.csv")) as report_file:
            write_csv_summary(report_file, summarize_sales(SALES))

        with atomic_write(run.path("README.txt")) as readme:
            readme.write("Validated sales report.\n")

    with require_files(destination, ["summary.csv", "README.txt"]):
        print("checking published output")


starting: validated report
publishing completed files
finished: validated report in 0.021073 seconds
checking published output
all required files exist


Now let's build one more context manager using the `@contextlib.contextmanager` decorator.

This is useful when a context manager is simple enough that writing a full class feels too verbose.

The code before `yield` is the enter phase.

The code after `yield` is the exit phase.

In [24]:
@contextlib.contextmanager
def section(title: str) -> Iterator[None]:
    print(f"\n--- {title} ---")
    try:
        yield
    finally:
        print(f"--- end: {title} ---\n")


Let's use it to make the output easier to read.

In [25]:
with section("summarize"):
    totals = summarize_sales(SALES)
    print(totals)

with section("write report"):
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "summary.csv"
        with atomic_write(path) as f:
            write_csv_summary(f, totals)
        print(path.read_text())


--- summarize ---
{'North': 1049.7, 'South': 1349.88, 'East': 524.79, 'West': 599.97}
--- end: summarize ---


--- write report ---
region,revenue
East,524.79
North,1049.70
South,1349.88
West,599.97

--- end: write report ---



Now let's put everything together in one clean project function.

The function will return the output directory so we can inspect the result.

In [26]:
def build_sales_report_project(rows: Iterable[dict], output_dir: Path) -> Path:
    rows = list(rows)
    output_dir = Path(output_dir)
    destination = output_dir / "published"

    with section("calculate totals"):
        totals = summarize_sales(rows)

    with section("run publishing pipeline"):
        with PublishingPipelineRun("sales report project", destination, mode="production") as run:
            with atomic_write(run.path("summary.csv")) as report_file:
                write_csv_summary(report_file, totals)

            with atomic_write(run.path("metadata.txt")) as metadata:
                metadata.write("project: sales report\n")
                metadata.write(f"rows: {len(rows)}\n")
                metadata.write(f"mode: {os.environ.get('REPORT_MODE')}\n")

    with section("validate output"):
        with require_files(destination, ["summary.csv", "metadata.txt"]):
            pass

    return destination


Let's run the final project solution.

In [27]:
with tempfile.TemporaryDirectory() as tmp:
    output = build_sales_report_project(SALES, Path(tmp))

    print("final files:", sorted(p.name for p in output.iterdir()))
    print("\nsummary.csv")
    print((output / "summary.csv").read_text())
    print("metadata.txt")
    print((output / "metadata.txt").read_text())



--- calculate totals ---
--- end: calculate totals ---


--- run publishing pipeline ---
starting: sales report project
publishing completed files
finished: sales report project in 0.024300 seconds
--- end: run publishing pipeline ---


--- validate output ---
all required files exist
--- end: validate output ---

final files: ['metadata.txt', 'summary.csv']

summary.csv
region,revenue
East,524.79
North,1049.70
South,1349.88
West,599.97

metadata.txt
project: sales report
rows: 6
mode: production



Let's add a few assertions.

Assertions are not a replacement for a full test suite, but they are very useful in notebooks.

They help confirm that the solution really behaves the way we expect.

In [28]:
with tempfile.TemporaryDirectory() as tmp:
    output = build_sales_report_project(SALES, Path(tmp))

    assert (output / "summary.csv").exists()
    assert (output / "metadata.txt").exists()

    summary_text = (output / "summary.csv").read_text()
    metadata_text = (output / "metadata.txt").read_text()

    assert "region,revenue" in summary_text
    assert "North" in summary_text
    assert "project: sales report" in metadata_text

print("all checks passed")


--- calculate totals ---
--- end: calculate totals ---


--- run publishing pipeline ---
starting: sales report project
publishing completed files
finished: sales report project in 0.013672 seconds
--- end: run publishing pipeline ---


--- validate output ---
all required files exist
--- end: validate output ---

all checks passed


### What this project demonstrated

This project used context managers for several different jobs:

* `Timer` measured a block of code
* `temporary_env` changed and restored an environment variable
* `change_dir` changed and restored the working directory
* `atomic_write` protected files from partial writes
* `ExitStack` managed multiple context managers dynamically
* `PipelineRun` grouped many resources into one higher-level context manager
* `PublishingPipelineRun` committed outputs only on success
* `require_files` validated output after a block finished
* `@contextlib.contextmanager` created a compact context manager function

The main idea is simple:

Context managers are not only for files.

They are a clean way to make setup and cleanup automatic, reliable, and readable.

### Extra challenge ideas

Try extending this project with one or more of these improvements:

1. Add JSON output in addition to CSV output
2. Add a rollback log when publishing fails
3. Add a `dry_run=True` mode that does not publish files
4. Add a context manager that temporarily redirects `stdout` to a log file
5. Add an async context manager version for an async pipeline
6. Turn the assertion cells into `pytest` tests
7. Add type hints to every public method
8. Add a context manager that temporarily patches a dictionary
9. Add a nested `ExitStack` for separate input and output resources
10. Add a project configuration object using `dataclasses`